<a href="https://colab.research.google.com/github/Liu-Xinran6/KUL-Hackathon-2026_P.S./blob/main/20260401_YOLOV8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install ultralytics -q

import ultralytics
ultralytics.checks()

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 44.0/112.6 GB disk)


In [9]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="A38n96YDUKB5e6OE5Kq3")
project = rf.workspace("xinranls-workspace").project("map-test-260331")
version = project.version(1)
dataset = version.download("yolov8")

print(f"Dataset downloaded to {dataset.location}")

loading Roboflow workspace...
loading Roboflow project...
Dataset downloaded to /content/Map-Test-260331-1


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s-seg.pt')

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    project='Historical_Map_Project',
    name='v1_train_run'
)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Map-Test-260331-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v1_train_run2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

In [ ]:
from ultralytics import YOLO
import glob
from IPython.display import Image, display

model = YOLO('/content/runs/segment/Historical_Map_Project/v1_train_run/weights/best.pt')

results = model.predict(
    source='/content/test_map.png',
    save=True,
    conf=0.5,
    project='Historical_Map_Project',
    name='predict_results'
)

predicted_image_path = glob.glob('/content/runs/segment/Historical_Map_Project/predict_results/*.jpg')[0]
display(Image(filename=predicted_image_path))

In [ ]:
from ultralytics import YOLO
import json
import uuid
import datetime
import os

# 1. 唤醒模型
model = YOLO('/content/runs/segment/Historical_Map_Project/v1_train_run/weights/best.pt')

# 2. 图片路径
image_path = '/content/test_map.png'
image_filename = os.path.basename(image_path)
results = model.predict(source=image_path, conf=0.5)

immarkus_annotations = []
# 获取当前时间，伪装成人类标注的时间戳
current_time = datetime.datetime.utcnow().isoformat()[:-3] + "Z"

# 3. 提取并“翻译”成 IMMARKUS 格式
if results[0].masks is not None:
    masks = results[0].masks.xy

    for i, polygon in enumerate(masks):
        # 核心魔法：把坐标列表拼成 SVG 要求的 "x1,y1 x2,y2" 格式
        points_str = " ".join([f"{x},{y}" for x, y in polygon])
        svg_value = f'<svg><polygon points="{points_str}" /></svg>'

        # 构建完全符合 IMMARKUS 规范的字典
        annotation = {
            "@context": "http://www.w3.org/ns/anno.jsonld",
            "id": str(uuid.uuid4()),  # 自动生成唯一ID
            "type": "Annotation",
            "target": {
                "source": image_filename,  # 必须和底图文件名一致
                "type": "SpecificResource",
                "selector": {
                    "type": "SvgSelector",
                    "value": svg_value
                }
            },
            "body": [
                {
                    "id": str(uuid.uuid4()),
                    "type": "Dataset",
                    "purpose": "classifying",
                    "source": "building", # 默认类别设为 building
                    "properties": {
                        "name": f"AI_提取地标_{i+1}" # 给它起个名字
                    },
                    "created": current_time,
                    "modified": current_time
                }
            ],
            "created": current_time,
            "creator": {
                "isGuest": True,
                "id": "AI_YOLOv8_Bot" # 署名：你的 AI 助手
            }
        }
        immarkus_annotations.append(annotation)

# 4. 导出为最终文件
json_filename = 'AI_to_IMMARKUS_ready.json'
with open(json_filename, 'w', encoding='utf-8') as f:
    json.dump(immarkus_annotations, f, ensure_ascii=False, indent=2)

print(f"共生成 {len(immarkus_annotations)} 个 IMMARKUS 兼容标注")